# TARGE — Prismatic VLM (Colab)

End-to-end pipeline: setup → install → login → paths → data → overview → train → eval.


## 1. Setup: GPU check + mount Drive


In [ ]:
print("== GPU ==")
!nvidia-smi -L || echo "(no GPU detected)"
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true

print("\n== Drive ==")
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


## 2. Install dependencies


In [ ]:
import time, sys
%cd /content/drive/MyDrive/targe-prismatic-vlms
print("[install] pip install -e . (this can take ~1-2 min) ...")
t0 = time.time()
!pip install -e . --quiet
print(f"[install] done in {time.time()-t0:.1f}s")
# !pip install flash-attn --no-build-isolation --quiet  # optional speedup on A100/L4


## 3. Logins (Hugging Face, W&B)


In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login, whoami

# Hugging Face
hf_token = userdata.get('hf_token')
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
who = whoami()
print(f"[hf] logged in as: {who.get('name')}  (orgs: {[o['name'] for o in who.get('orgs', [])]})")

# W&B (optional)
try:
    wandb_token = userdata.get('wandb_token')
    os.environ["WANDB_API_KEY"] = wandb_token
    print("[wandb] WANDB_API_KEY set from Colab secrets")
except Exception:
    print("[wandb] no wandb_token secret found — skipping (training will not log to W&B)")


## 4. Configure filepaths

All paths live here. Drive = persistent (slow); local = fast scratch on the Colab VM.


In [ ]:
import os
from pathlib import Path

# --- Drive (persistent) — write-only target for outputs ---
DRIVE_ROOT = Path("/content/drive/MyDrive/targe-prismatic-vlms")  # repo + outputs
DRIVE_RUNS = DRIVE_ROOT / "runs"                                  # checkpoints + train.log + eval results

# --- Local (Colab VM SSD) — data lives here, never copied back ---
TAR_PATH       = Path("/content/train_subset_5k.tar")             # input tarball (already on VM)
DATASET_ROOT   = Path("/content/data")                            # value passed as --dataset.dataset_root_dir
DATA_DIR       = DATASET_ROOT / "download/llava-laion-cc-sbu-558k"  # layout LLaVa_V15_Config expects
CHAT_JSON      = DATA_DIR / "chat.json"                           # filename LLaVa_V15_Config expects

# HF cache: keep on VM, not Drive
HF_CACHE = Path("/content/hf_cache")
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"]      = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")

DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

print(f"REPO (Drive)   : {DRIVE_ROOT}")
print(f"RUNS (Drive)   : {DRIVE_RUNS}")
print(f"TAR    (local) : {TAR_PATH}   ({'exists' if TAR_PATH.exists() else 'MISSING'})")
print(f"DATA   (local) : {DATA_DIR}   ({'extracted' if DATA_DIR.exists() else 'not yet extracted'})")
print(f"HF cache       : {HF_CACHE}")


## 5. Untar dataset into the layout `llava-v15` expects

Extracts to `/content/data/download/llava-laion-cc-sbu-558k/` and renames the chat json to `chat.json` so `LLaVa_V15_Config` finds everything at its default paths (relative to a `--dataset.dataset_root_dir` we override to `/content/data`). No program-side changes needed.


In [ ]:
import time, tarfile, shutil

OLD_DIR = Path("/content/train_subset_5k")        # where an earlier attempt may have left it

assert TAR_PATH.exists() or OLD_DIR.exists(), (
    f"Neither {TAR_PATH} nor {OLD_DIR} exists. Upload the tarball to {TAR_PATH}."
)

if CHAT_JSON.exists():
    n_files = sum(1 for _ in DATA_DIR.rglob("*") if _.is_file())
    print(f"[skip] {DATA_DIR} already populated ({n_files:,} files)")
else:
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)

    if OLD_DIR.exists() and any(OLD_DIR.iterdir()):
        # Reuse what's already on local disk — atomic rename, no copy
        print(f"[move] {OLD_DIR} -> {DATA_DIR}")
        DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
        if DATA_DIR.exists():
            DATA_DIR.rmdir()  # only succeeds if empty; we ensured CHAT_JSON missing above
        OLD_DIR.rename(DATA_DIR)
    else:
        # Fresh extraction. Peek to decide whether to --strip-components 1
        with tarfile.open(TAR_PATH) as tf:
            tops = set()
            for i, m in enumerate(tf):
                tops.add(m.name.split("/", 1)[0])
                if i > 20:
                    break
        has_wrapper = len(tops) == 1 and next(iter(tops)) not in {"", "."}
        strip_flag = "--strip-components 1" if has_wrapper else ""
        print(f"[tar] top-level sample : {sorted(tops)[:5]}{'...' if len(tops) > 5 else ''}")
        print(f"[tar] strip wrapper dir? {has_wrapper}")

        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[tar] extracting -> {DATA_DIR}  ({TAR_PATH.stat().st_size / 1e9:.2f} GB)")
        t0 = time.time()
        !tar -xf "{TAR_PATH}" {strip_flag} -C "{DATA_DIR}"
        print(f"[tar] done in {time.time()-t0:.1f}s")

    # Rename chat_subset_5k.json -> chat.json in place (no copy) so LLaVa_V15_Config finds it
    src_json = DATA_DIR / "chat_subset_5k.json"
    if src_json.exists() and not CHAT_JSON.exists():
        src_json.rename(CHAT_JSON)
        print(f"[rename] {src_json.name} -> {CHAT_JSON.name}")

print(f"\n== Layout under {DATA_DIR} ==")
!ls -la "{DATA_DIR}" | grep -v "^total" | head -20

if not CHAT_JSON.exists():
    hits = list(Path("/content").rglob("chat.json")) + list(Path("/content").rglob("chat_subset_5k.json"))
    raise FileNotFoundError(f"Expected {CHAT_JSON} but missing. Found candidates: {hits}")

n_files = sum(1 for _ in DATA_DIR.rglob("*") if _.is_file())
print(f"\n[ok] {CHAT_JSON} ({CHAT_JSON.stat().st_size / 1e6:.2f} MB)")
print(f"[ok] {n_files:,} files under {DATA_DIR}")


## 6. Dataset overview


In [ ]:
import json
from collections import Counter

n_files = sum(1 for _ in DATA_DIR.rglob("*") if _.is_file())
n_imgs  = sum(1 for p in DATA_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
n_dirs  = sum(1 for _ in DATA_DIR.iterdir() if _.is_dir())

with open(CHAT_JSON) as f:
    chat = json.load(f)

n_entries    = len(chat)
n_with_image = sum(1 for e in chat if e.get("image"))
n_text_only  = n_entries - n_with_image
turns        = Counter(len(e.get("conversations", [])) for e in chat)

print(f"Data dir          : {DATA_DIR}")
print(f"Top-level folders : {n_dirs}")
print(f"Total files       : {n_files:,}")
print(f"  image files     : {n_imgs:,}")
print(f"Chat JSON         : {CHAT_JSON.name}")
print(f"  total entries   : {n_entries:,}")
print(f"  with image      : {n_with_image:,}")
print(f"  text-only       : {n_text_only:,}")
print(f"  turns/example   : {dict(sorted(turns.items()))}")
print("\nSplit             : train-only (no held-out val in this subset)")


### Inspect a sample by image ID

Pass any substring of an image filename (e.g. `"0074375"`) and the cell renders the image + its chat turns.


In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown
from PIL import Image

_chat_cache = {}
def _load_chat():
    if "chat" not in _chat_cache:
        with open(CHAT_JSON) as f:
            _chat_cache["chat"] = json.load(f)
        _chat_cache["by_img"] = {e.get("image", ""): e for e in _chat_cache["chat"] if e.get("image")}
    return _chat_cache["chat"], _chat_cache["by_img"]

def show_sample(image_id: str, max_thumb=512):
    """`image_id` can be the full image path inside DATA_DIR, a filename, or any substring like '0074375'."""
    _, by_img = _load_chat()
    matches = [k for k in by_img if image_id in k]
    if not matches:
        print(f"[miss] no chat entry whose image path contains {image_id!r}")
        return
    if len(matches) > 1:
        print(f"[note] {len(matches)} matches — showing first. Others: {matches[1:6]}{'...' if len(matches)>6 else ''}")
    key = matches[0]
    entry = by_img[key]
    img_path = DATA_DIR / key

    display(Markdown(f"**id:** `{entry.get('id','?')}`  &nbsp;|&nbsp; **image:** `{key}`"))
    if img_path.is_file():
        im = Image.open(img_path)
        w, h = im.size
        if max(w, h) > max_thumb:
            im.thumbnail((max_thumb, max_thumb))
        display(im)
        display(Markdown(f"<sub>{w}×{h}px · {img_path.stat().st_size/1024:.1f} KB · `{img_path}`</sub>"))
    else:
        print(f"[warn] image not found on disk: {img_path}")

    md = ["**Conversation:**"]
    for turn in entry.get("conversations", []):
        role = turn.get("from", "?")
        text = turn.get("value", "").replace("<image>", "_<image>_")
        md.append(f"- **{role}:** {text}")
    display(Markdown("\n".join(md)))

# Example
show_sample("0074375")


## 6b. Validate images & prune `chat.json`

PIL raises `UnidentifiedImageError` on truncated / corrupt / 0-byte files mid-training, crashing the dataloader. This cell verifies every image referenced by `chat.json` and rewrites it to drop bad entries. Original is backed up to `chat.original.json`. Idempotent: skips if already pruned.


In [ ]:
import json, shutil, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image, UnidentifiedImageError

BACKUP = CHAT_JSON.with_name("chat.original.json")

if BACKUP.exists():
    with open(BACKUP) as f:
        n_before = len(json.load(f))
    with open(CHAT_JSON) as f:
        n_after = len(json.load(f))
    print(f"[skip] chat.json already pruned (backup: {BACKUP.name})")
    print(f"  before : {n_before:,}")
    print(f"  after  : {n_after:,}")
    print(f"  dropped: {n_before - n_after:,}")
else:
    with open(CHAT_JSON) as f:
        chat = json.load(f)

    def check(e):
        img_rel = e.get("image")
        if not img_rel:
            return e, True, None     # text-only, keep
        p = DATA_DIR / img_rel
        if not p.is_file() or p.stat().st_size == 0:
            return e, False, "missing/empty"
        try:
            with Image.open(p) as im:
                im.verify()           # cheap header-level check
            return e, True, None
        except (UnidentifiedImageError, OSError, SyntaxError) as ex:
            return e, False, type(ex).__name__

    print(f"[verify] checking {len(chat):,} entries (32 threads) ...")
    t0 = time.time()
    good, bad = [], []
    with ThreadPoolExecutor(max_workers=32) as pool:
        futs = [pool.submit(check, e) for e in chat]
        for i, fut in enumerate(as_completed(futs), 1):
            e, ok, reason = fut.result()
            if ok:
                good.append(e)
            else:
                bad.append((e, reason))
            if i % 1000 == 0 or i == len(chat):
                print(f"  {i:>6,}/{len(chat):,}   bad so far: {len(bad):,}")

    dt = time.time() - t0
    print(f"\n[verify] done in {dt:.1f}s  ({len(chat)/max(dt,1):.0f} img/s)")
    print(f"  good   : {len(good):,}")
    print(f"  bad    : {len(bad):,}")
    if bad:
        from collections import Counter
        print(f"  reasons: {Counter(r for _, r in bad).most_common()}")
        print("  first 5 bad files:")
        for e, r in bad[:5]:
            print(f"    - [{r}] {e.get('image')}")

    shutil.copy2(CHAT_JSON, BACKUP)
    with open(CHAT_JSON, "w") as f:
        json.dump(good, f)
    print(f"\n[write] {CHAT_JSON} ({len(good):,} entries)")
    print(f"[backup] {BACKUP} ({len(chat):,} original entries)")


## 7. Train (align stage)


In [ ]:
import os, time
os.environ["PYTHONUNBUFFERED"] = "1"

RUN_ID   = "targe-smollm2-5k"
LOG_PATH = DRIVE_RUNS / RUN_ID / "train.log"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
WB_KEY_VAL   = os.environ.get("WANDB_API_KEY", "").strip()

# scripts/pretrain.py is installed editable, so it imports fine from any CWD.
# We launch from /content so LLaVa_V15_Config's default relative `data/download/...`
# resolves to /content/data/download/... (where we put the extracted files).
# We pass --run_root_dir as an absolute path so checkpoints still land on Drive.
REPO_DIR  = DRIVE_ROOT
LAUNCH_CWD = Path("/content")

print(f"[train] run_id     : {RUN_ID}")
print(f"[train] log file   : {LOG_PATH}")
print(f"[train] launch CWD : {LAUNCH_CWD}  (so prismatic's `data/` resolves to /content/data/)")
print(f"[train] data root  : {DATA_DIR}")
print(f"[train] runs root  : {DRIVE_RUNS}  (absolute -> Drive)")
print(f"[train] hf_token   : {'<set, ' + str(len(HF_TOKEN_VAL)) + ' chars>' if HF_TOKEN_VAL else '<empty — anonymous mode>'}")
print(f"[train] wb_key     : {'<set>' if WB_KEY_VAL else '<empty — wandb disabled>'}")
print(f"[train] started    : {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

TRACKERS = "[jsonl,wandb]" if WB_KEY_VAL else "[jsonl]"
PRETRAIN = REPO_DIR / "scripts" / "pretrain.py"

%cd /content
!set -o pipefail; HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" WANDB_API_KEY="{WB_KEY_VAL}" HF_HOME="/content/hf_cache" \
  stdbuf -oL -eL torchrun --standalone --nnodes 1 --nproc-per-node 1 "{PRETRAIN}" \
    --model.type "targe-smollm2-360m-clipb-224px" \
    --stage align \
    --model.align_per_device_batch_size 14 \
    --model.align_global_batch_size 14 \
    --model.align_learning_rate 1e-4 \
    --model.align_max_steps 500 \
    --dataset.type "llava-v15" \
    --run_id "{RUN_ID}" \
    --run_root_dir "{DRIVE_RUNS}" \
    --hf_token HF_TOKEN \
    --trackers '{TRACKERS}' \
    --wandb_entity nbusich-northeastern-university \
    --wandb_project targe 2>&1 | tee -a "{LOG_PATH}"

print(f"\n[train] finished : {time.strftime('%Y-%m-%d %H:%M:%S')}")


## 8. Eval (interactive generation against a checkpoint)


In [ ]:
RUN_ID   = "targe-smollm2-5k"
CKPT_DIR = DRIVE_RUNS / RUN_ID

ckpts = sorted((CKPT_DIR / "checkpoints").glob("*.pt"))
print(f"[eval] run dir   : {CKPT_DIR}")
print(f"[eval] checkpoints found ({len(ckpts)}):")
for c in ckpts:
    sz = c.stat().st_size / 1e9
    print(f"   - {c.name:40s}  {sz:6.2f} GB")

assert ckpts, f"No checkpoints found in {CKPT_DIR / 'checkpoints'}"
LATEST = ckpts[-1]
print(f"\n[eval] using latest: {LATEST.name}")


In [ ]:
%cd /content/drive/MyDrive/targe-prismatic-vlms
HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
!HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" HF_HOME="/content/hf_cache" \
  python scripts/generate.py --model_path "{CKPT_DIR}" --hf_token HF_TOKEN


## 9. Six-group ablation sweep

Six configurations, **one trained checkpoint**. Toggling bypass flags through `SelectorCompressorPipeline` isolates the contributions of the Selector and the Q-Former without retraining.

| Group | route_mode      | use_qformer | Notes |
|-------|-----------------|-------------|-------|
| A     | full            | False       | upper bound: all N tokens, no compression |
| B     | random_topk     | False       | lower bound: random k tokens |
| C     | oracle          | False       | indices precomputed from LLM cross-attention |
| D     | selector        | True        | trained behavior (status quo) |
| E     | selector        | False       | selector-only |
| F     | full            | True        | Q-Former on all N tokens |


In [ ]:
import json, random
from pathlib import Path

HELDOUT_JSON = DATA_DIR / "chat_heldout.json"
N_HELDOUT = 200

if HELDOUT_JSON.exists():
    with open(CHAT_JSON) as f:
        n_train = len(json.load(f))
    with open(HELDOUT_JSON) as f:
        n_held = len(json.load(f))
    print(f"[skip] heldout already created  train={n_train:,}  heldout={n_held:,}")
else:
    with open(CHAT_JSON) as f:
        chat = json.load(f)
    rng = random.Random(7)
    rng.shuffle(chat)
    held = chat[:N_HELDOUT]
    train = chat[N_HELDOUT:]
    json.dump(held,  open(HELDOUT_JSON, "w"))
    json.dump(train, open(CHAT_JSON,    "w"))
    print(f"[write] {HELDOUT_JSON.name}: {len(held):,}")
    print(f"[write] {CHAT_JSON.name}:    {len(train):,}")


### 9.1 Precompute oracle indices

For each held-out example, run a forward with `route_mode="full"` + `use_qformer=False` so the LLM sees every projected visual token. Then average attention from response-token positions to visual-token positions across the early layers and take top-k. Saved as `oracle_indices.pt`.


In [ ]:
RUN_ID   = "targe-smollm2-5k"
CKPT_DIR = DRIVE_RUNS / RUN_ID
ORACLE_PT = CKPT_DIR / "oracle_indices.pt"

%cd /content/drive/MyDrive/targe-prismatic-vlms
HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
!HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" HF_HOME="/content/hf_cache" \
  python scripts/eval/precompute_oracle.py \
    --model_path "{CKPT_DIR}" \
    --heldout_json "{HELDOUT_JSON}" \
    --image_root "{DATA_DIR}" \
    --out_pt "{ORACLE_PT}" \
    --hf_token HF_TOKEN


### 9.2 Run the 6-group sweep

For each held-out example, runs all six routing configurations against the same checkpoint, capturing generations, connector-output latents (for cosine similarity vs. Group A), Oracle IoU (groups D, E), and per-group hardware costs (FLOPS via `fvcore` if available, latency via CUDA events). If `pope_json` is supplied, also runs POPE-style yes/no accuracy.


In [ ]:
RESULTS_JSON = CKPT_DIR / "ablation_results.json"
# Optional: drop a POPE-format JSON (list of {image, question, label}) at this path and uncomment.
POPE_JSON = ""  # e.g. f"{CKPT_DIR}/pope_subset.json"

%cd /content/drive/MyDrive/targe-prismatic-vlms
HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
POPE_ARG = f'--pope_json "{POPE_JSON}"' if POPE_JSON else ""
!HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" HF_HOME="/content/hf_cache" \
  python scripts/eval/run_ablation.py \
    --model_path "{CKPT_DIR}" \
    --heldout_json "{HELDOUT_JSON}" \
    --image_root "{DATA_DIR}" \
    --oracle_pt "{ORACLE_PT}" \
    --out_json "{RESULTS_JSON}" \
    {POPE_ARG} \
    --hf_token HF_TOKEN


### 9.3 Results table


In [ ]:
import json
import pandas as pd

with open(RESULTS_JSON) as f:
    res = json.load(f)

rows = []
for name, s in res["summary"].items():
    hw = s.get("hardware") or {}
    pope = s.get("pope") or {}
    rows.append({
        "group":             name,
        "n":                 s.get("n_generations"),
        "cos_vs_A":          s.get("cos_vs_A_mean"),
        "iou_vs_oracle":     s.get("iou_vs_oracle_mean"),
        "pope_acc":          pope.get("accuracy"),
        "pope_yes_rate":     pope.get("yes_rate"),
        "flops":             hw.get("flops"),
        "latency_ms_median": hw.get("latency_ms_median"),
    })
df = pd.DataFrame(rows).set_index("group")
print(df.to_string(float_format=lambda v: f"{v:.4f}" if isinstance(v, float) else str(v)))
